# Act 3 — Hidden relationships

> **Opening question:** *"Everyone assumes bigger = more expensive. But is GrLivArea really the strongest link to SalePrice — or does something less obvious secretly pull more weight?"*

---
This act uses correlation analysis and scatter plots to reveal which features
move in lockstep with SalePrice — and which ones are red herrings.

In [ ]:
import sys
sys.path.append('..')
from src.utils import *

act_header(
    act_num=3,
    title="Hidden relationships",
    opening_question="Which features are most tightly linked to SalePrice — and which links surprise us?"
)

df = load_processed('cleaned.csv')

## 3.1 — Correlation heatmap
How closely does each feature travel with SalePrice?
A score near +1 means 'always rises together'. Near 0 means 'no relationship'.

In [ ]:
plot_correlation_heatmap(df, figsize=(13, 10))

## 3.2 — Top 10 features correlated with SalePrice

In [ ]:
numeric_df = df.select_dtypes(include='number')
corr_with_price = numeric_df.corr()['SalePrice'].drop('SalePrice').sort_values(ascending=False)

top10  = corr_with_price.head(10)
bot5   = corr_with_price.tail(5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top positive
bars = axes[0].barh(top10.index[::-1], top10.values[::-1], color=PALETTE['purple'], alpha=0.85)
axes[0].set_title('Top 10 features most linked to SalePrice', fontsize=13)
axes[0].set_xlabel('Correlation with SalePrice')
axes[0].axvline(0, color='gray', linewidth=0.5)
for bar, val in zip(bars, top10.values[::-1]):
    axes[0].text(val + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9)

# Bottom (negative / weak)
colors_neg = [PALETTE['coral'] if v < 0 else PALETTE['gray'] for v in bot5.values]
axes[1].barh(bot5.index, bot5.values, color=colors_neg, alpha=0.85)
axes[1].set_title('Features with weakest / negative link', fontsize=13)
axes[1].set_xlabel('Correlation with SalePrice')
axes[1].axvline(0, color='gray', linewidth=0.5)

plt.tight_layout()
save_figure(fig, 'act3_top_correlations.png')
plt.show()

top_feature = top10.index[0]
top_val = top10.values[0]
insight_callout(
    f"'{top_feature}' has the strongest link to SalePrice (r = {top_val:.2f}).\n"
    "Correlation of 0.8+ means these two almost always move together.\n"
    "GrLivArea (living area) is strong too — but quality beats size as the #1 predictor.",
    label="The surprise finding"
)

## 3.3 — OverallQual vs SalePrice (the strongest relationship)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

sns.boxplot(
    data=df, x='OverallQual', y='SalePrice',
    palette=sns.color_palette('BuPu', 10),
    ax=ax, width=0.6,
    flierprops=dict(marker='o', markerfacecolor=PALETTE['coral'], markersize=3, alpha=0.4)
)
ax.set_title('Quality score vs sale price — a near-perfect staircase', fontsize=13)
ax.set_xlabel('Overall quality (1 = Poor, 10 = Excellent)')
ax.set_ylabel('Sale price')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
save_figure(fig, 'act3_qual_vs_price.png')
plt.show()

## 3.4 — GrLivArea vs SalePrice (size vs price)

In [ ]:
import plotly.express as px

fig = px.scatter(
    df, x='GrLivArea', y='SalePrice',
    color='OverallQual',
    color_continuous_scale='Purples',
    hover_data=['Neighborhood', 'YearBuilt'] if 'Neighborhood' in df.columns else [],
    title='Living area vs sale price — coloured by quality score',
    labels={'GrLivArea': 'Above-ground living area (sq ft)', 'SalePrice': 'Sale price ($)'},
    template='plotly_white',
    opacity=0.65
)
fig.update_layout(
    title_font_size=14,
    coloraxis_colorbar_title='Quality',
    margin=dict(t=60, b=40)
)
save_plotly(fig, 'act3_size_vs_price_interactive.html')
fig.show()

In [ ]:
insight_callout(
    "The scatter shows a clear upward trend — bigger homes cost more.\n"
    "But look at the colour: at the same square footage, a quality-9 home\n"
    "can cost 50-80% more than a quality-5 home of identical size.\n"
    "Size opens the door. Quality sets the price.",
    label="Size vs quality — who wins?"
)

## 3.5 — Neighbourhood effect
Same size, same quality — but does the street address still change the price?

In [ ]:
if 'Neighborhood' in df.columns:
    nbr_median = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(13, 6))
    colors_nbr = [PALETTE['purple'] if v > df['SalePrice'].median() else PALETTE['gray'] for v in nbr_median.values]
    ax.barh(nbr_median.index[::-1], nbr_median.values[::-1] / 1000, color=colors_nbr[::-1], alpha=0.85)
    ax.axvline(df['SalePrice'].median() / 1000, color=PALETTE['coral'], linestyle='--', lw=1.5, label='Overall median')
    ax.set_xlabel('Median sale price ($k)')
    ax.set_title('Median house price by neighbourhood (purple = above overall median)', fontsize=13)
    ax.legend(fontsize=10)
    plt.tight_layout()
    save_figure(fig, 'act3_neighborhood_prices.png')
    plt.show()

    top_nbr    = nbr_median.index[0]
    bottom_nbr = nbr_median.index[-1]
    insight_callout(
        f"'{top_nbr}' is the most expensive neighbourhood (median ${nbr_median.iloc[0]/1000:.0f}k).\n"
        f"'{bottom_nbr}' is the most affordable (median ${nbr_median.iloc[-1]/1000:.0f}k).\n"
        f"That's a ${(nbr_median.iloc[0] - nbr_median.iloc[-1])/1000:.0f}k gap for homes that might be\n"
        "identical in size and quality — pure location premium.",
        label="The location premium"
    )

---
## Act 3 — Closing punchline

In [ ]:
punchline(
    "OverallQual is the #1 driver of SalePrice — beating even living area. "
    "Size matters, but quality multiplies it. "
    "And neighbourhood can add or subtract hundreds of thousands regardless of the house itself. "
    "In Act 4, we follow how prices evolved across time."
)

**Next → [Act 4 — Change Over Time](04_act4_time_trends.ipynb)**